# Day 18 - Peer Comparison & Percentile Ranking

## Objectives

- Inspect company classification data
- Identify available peer group information
- Load latest financial metrics
- Compare companies within peer groups
- Calculate peer percentile rankings
- Generate peer-wise company ranks
- Identify top and bottom peer performers
- Export peer comparison reports

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import os

In [2]:
conn = sqlite3.connect("../database/nifty100.db")

print("Database Connected Successfully")

Database Connected Successfully


In [3]:
companies = pd.read_sql(
    "SELECT * FROM companies",
    conn
)

print("Companies Table Shape:", companies.shape)

companies.head()

Companies Table Shape: (93, 12)


,mkt_fintech_—_nifty_100__|__companies__|__92_records,unnamed:_1,unnamed:_2,unnamed:_3,unnamed:_4,unnamed:_5,unnamed:_6,unnamed:_7,unnamed:_8,unnamed:_9,unnamed:_10,unnamed:_11
0,id,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,roce_percentage,roe_percentage
1,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10,1657,46,34.9
2,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10,175,9,8.59
3,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1,363,11.6,13.64
4,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10,67,96.5,14.7


In [4]:
companies.columns = companies.iloc[0]

companies = (
    companies.iloc[1:]
    .reset_index(drop=True)
)

companies.head()

,id,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,roce_percentage,roe_percentage
0,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10,1657,46,34.9
1,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10,175,9,8.59
2,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1,363,11.6,13.64
3,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10,67,96.5,14.7
4,ADANIPORTS,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Ports & Special Economic Zone Ltd\n,https://in.tradingview.com/chart/?symbol=NSE%3...,Adani Ports & Special Economic Zone is in the ...,http://www.adaniports.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,2,265,12.9,18.1


In [5]:
print("Companies Table Columns:")

for col in companies.columns:
    print(col)

Companies Table Columns:
id
company_logo
company_name
chart_link
about_company
website
nse_profile
bse_profile
face_value
book_value
roce_percentage
roe_percentage


In [6]:
peer_columns = [
    col
    for col in companies.columns
    if any(
        word in str(col).lower()
        for word in [
            "sector",
            "industry",
            "category",
            "peer"
        ]
    )
]

print("Possible Peer Group Columns:")
print(peer_columns)

Possible Peer Group Columns:
[]


In [7]:
print("All Companies Columns:")
print(companies.columns.tolist())

All Companies Columns:
['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage']


In [8]:
companies.head(10)

,id,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,roce_percentage,roe_percentage
0,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10,1657,46,34.9
1,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10,175,9,8.59
2,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1,363,11.6,13.64
3,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10,67,96.5,14.7
4,ADANIPORTS,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Ports & Special Economic Zone Ltd\n,https://in.tradingview.com/chart/?symbol=NSE%3...,Adani Ports & Special Economic Zone is in the ...,http://www.adaniports.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,2,265,12.9,18.1
5,ADANIPOWER,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Power Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Power (APL), a part of the diversified A...",http://www.adani.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10,145,32.2,57.1
6,AMBUJACEM,https://mkt.in/static/mkt-icons/nifty100/AMBUJ...,Ambuja Cements Ltd,https://in.tradingview.com/chart/qGsydD2w/?sym...,Ambuja Cements Ltd. is among the leading cemen...,https://www.ambujacement.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/amb...,2,205,12.8,9.24
7,APOLLOHOSP,https://mkt.in/static/mkt-icons/nifty100/APOLL...,Apollo Hospitals\nChain of Indian private hosp...,https://in.tradingview.com/chart/?symbol=NSE%3...,Apollo Hospitals Enterprise Limited is an Indi...,https://www.apollohospitals.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/apo...,5,522,17.2,13.81
8,ASIANPAINT,https://mkt.in/static/mkt-icons/nifty100/ASIAN...,Asian Paints\nIndian Multi-National Paint and ...,https://in.tradingview.com/chart/?symbol=NSE%3...,Asian Paints Ltd is an Indian multinational pa...,https://www.asianpaints.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/asi...,1,188,41.75,31.45
9,ATGL,https://mkt.in/static/mkt-icons/nifty100/ATGL.png,Adani Total Gas Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,AGL is engaged in City Gas Distribution (CGD) ...,http://www.adanigas.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1,36,21.2,20.5


In [9]:
tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

print("Available Database Tables:")
print(tables)

Available Database Tables:
            name
0      companies
1       analysis
2   balancesheet
3       cashflow
4  profitandloss
5      documents
6    prosandcons


In [10]:
for table_name in tables["name"]:
    print("\n==============================")
    print("TABLE:", table_name)
    
    temp = pd.read_sql(
        f'SELECT * FROM "{table_name}" LIMIT 5',
        conn
    )
    
    print("Columns:")
    print(temp.columns.tolist())


TABLE: companies
Columns:
['mkt_fintech_—_nifty_100__|__companies__|__92_records', 'unnamed:_1', 'unnamed:_2', 'unnamed:_3', 'unnamed:_4', 'unnamed:_5', 'unnamed:_6', 'unnamed:_7', 'unnamed:_8', 'unnamed:_9', 'unnamed:_10', 'unnamed:_11']

TABLE: analysis
Columns:
['bluestock_fintech_—_nifty_100__|__analysis__|__20_records', 'unnamed:_1', 'unnamed:_2', 'unnamed:_3', 'unnamed:_4', 'unnamed:_5']

TABLE: balancesheet
Columns:
['bluestock_fintech_—_nifty_100__|__balance_sheet__|__1,312_records', 'unnamed:_1', 'unnamed:_2', 'unnamed:_3', 'unnamed:_4', 'unnamed:_5', 'unnamed:_6', 'unnamed:_7', 'unnamed:_8', 'unnamed:_9', 'unnamed:_10', 'unnamed:_11', 'unnamed:_12']

TABLE: cashflow
Columns:
['bluestock_fintech_—_nifty_100__|__cash_flow__|__1,187_records', 'unnamed:_1', 'unnamed:_2', 'unnamed:_3', 'unnamed:_4', 'unnamed:_5', 'unnamed:_6']

TABLE: profitandloss
Columns:
['bluestock_fintech_—_nifty_100__|__profit_&_loss__|__1,276_records', 'unnamed:_1', 'unnamed:_2', 'unnamed:_3', 'unnamed:_4'

In [11]:
sector_report = pd.read_csv(
    "../reports/sector_distribution.csv"
)

peer_report = pd.read_csv(
    "../reports/peer_group_distribution.csv"
)

print("SECTOR REPORT")
print("Shape:", sector_report.shape)
print("Columns:", sector_report.columns.tolist())

display(sector_report.head(10))

print("\nPEER GROUP REPORT")
print("Shape:", peer_report.shape)
print("Columns:", peer_report.columns.tolist())

display(peer_report.head(10))

SECTOR REPORT
Shape: (10, 2)
Columns: ['broad_sector', 'count']


,broad_sector,count
0,Financials,23
1,Energy,14
2,Consumer Discretionary,14
3,Industrials,10
4,Materials,9
5,Consumer Staples,7
6,Healthcare,6
7,Information Technology,5
8,Communication Services,2
9,Real Estate,2



PEER GROUP REPORT
Shape: (11, 2)
Columns: ['peer_group_name', 'count']


,peer_group_name,count
0,Automobiles,7
1,Power & Utilities,7
2,FMCG,7
3,Private Banks,5
4,IT Services,5
5,Pharmaceuticals,5
6,Oil & Gas,5
7,Public Sector Banks,4
8,Life Insurance,4
9,Steel,4


In [12]:
classification_matches = []

for table_name in tables["name"]:
    temp = pd.read_sql(
        f'SELECT * FROM "{table_name}" LIMIT 5',
        conn
    )

    for col in temp.columns:
        col_name = str(col).lower()

        if any(
            keyword in col_name
            for keyword in [
                "sector",
                "industry",
                "peer",
                "broad"
            ]
        ):
            classification_matches.append(
                {
                    "table": table_name,
                    "column": col
                }
            )

classification_matches = pd.DataFrame(
    classification_matches
)

print("Classification Columns Found:")

classification_matches

Classification Columns Found:


""


In [13]:
import glob

csv_files = glob.glob(
    "../**/*.csv",
    recursive=True
)

mapping_candidates = []

for file_path in csv_files:
    try:
        temp = pd.read_csv(
            file_path,
            nrows=5
        )

        matching_cols = [
            col
            for col in temp.columns
            if any(
                keyword in str(col).lower()
                for keyword in [
                    "broad_sector",
                    "peer_group",
                    "sector",
                    "industry"
                ]
            )
        ]

        if matching_cols:
            mapping_candidates.append(
                {
                    "file": file_path,
                    "columns": matching_cols
                }
            )

    except Exception:
        pass

print("Possible Mapping Files:")

for item in mapping_candidates:
    print(
        "\nFile:",
        item["file"]
    )

    print(
        "Columns:",
        item["columns"]
    )

Possible Mapping Files:

File: ..\reports\benchmark_companies.csv
Columns: ['peer_group_name']

File: ..\reports\benchmark_companies_day7.csv
Columns: ['peer_group_name']

File: ..\reports\peer_group_distribution.csv
Columns: ['peer_group_name']

File: ..\reports\sector_distribution.csv
Columns: ['broad_sector']


In [14]:
for item in mapping_candidates:
    file_path = item["file"]

    try:
        temp = pd.read_csv(
            file_path,
            nrows=5
        )

        company_cols = [
            col
            for col in temp.columns
            if any(
                keyword in str(col).lower()
                for keyword in [
                    "company",
                    "symbol",
                    "ticker",
                    "name"
                ]
            )
        ]

        if company_cols:
            print("\n================================")
            print("POSSIBLE COMPANY MAPPING FILE")
            print("File:", file_path)
            print("Company Columns:", company_cols)
            print(
                "Classification Columns:",
                item["columns"]
            )

            display(temp)
            
    except Exception:
        pass


POSSIBLE COMPANY MAPPING FILE
File: ..\reports\benchmark_companies.csv
Company Columns: ['peer_group_name', 'company_id']
Classification Columns: ['peer_group_name']


,id,peer_group_name,company_id,is_benchmark
0,1,Private Banks,HDFCBANK,True
1,6,Public Sector Banks,SBIN,True
2,10,IT Services,TCS,True
3,15,Pharmaceuticals,SUNPHARMA,True
4,20,Automobiles,MARUTI,True



POSSIBLE COMPANY MAPPING FILE
File: ..\reports\benchmark_companies_day7.csv
Company Columns: ['peer_group_name', 'company_id']
Classification Columns: ['peer_group_name']


,id,peer_group_name,company_id,is_benchmark
0,1,Private Banks,HDFCBANK,True
1,6,Public Sector Banks,SBIN,True
2,10,IT Services,TCS,True
3,15,Pharmaceuticals,SUNPHARMA,True
4,20,Automobiles,MARUTI,True



POSSIBLE COMPANY MAPPING FILE
File: ..\reports\peer_group_distribution.csv
Company Columns: ['peer_group_name']
Classification Columns: ['peer_group_name']


,peer_group_name,count
0,Automobiles,7
1,Power & Utilities,7
2,FMCG,7
3,Private Banks,5
4,IT Services,5


In [17]:
import pandas as pd

peer_groups = pd.read_excel(
    "../data/raw/supporting/peer_groups.xlsx"
)

sectors = pd.read_excel(
    "../data/raw/supporting/sectors.xlsx"
)

print("PEER GROUPS")
print("Shape:", peer_groups.shape)
print("Columns:", peer_groups.columns.tolist())
display(peer_groups.head(10))

print("\nSECTORS")
print("Shape:", sectors.shape)
print("Columns:", sectors.columns.tolist())
display(sectors.head(10))

PEER GROUPS
Shape: (56, 4)
Columns: ['id', 'peer_group_name', 'company_id', 'is_benchmark']


,id,peer_group_name,company_id,is_benchmark
0,1,Private Banks,HDFCBANK,True
1,2,Private Banks,ICICIBANK,False
2,3,Private Banks,AXISBANK,False
3,4,Private Banks,KOTAKBANK,False
4,5,Private Banks,INDUSINDBK,False
5,6,Public Sector Banks,SBIN,True
6,7,Public Sector Banks,BANKBARODA,False
7,8,Public Sector Banks,CANBK,False
8,9,Public Sector Banks,PNB,False
9,10,IT Services,TCS,True



SECTORS
Shape: (92, 6)
Columns: ['id', 'company_id', 'broad_sector', 'sub_sector', 'index_weight_pct', 'market_cap_category']


,id,company_id,broad_sector,sub_sector,index_weight_pct,market_cap_category
0,1,ABB,Industrials,Capital Goods,0.81,Large Cap
1,2,ADANIENSOL,Energy,Power & Utilities,0.65,Large Cap
2,3,ADANIENT,Industrials,Conglomerates,1.50,Large Cap
3,4,ADANIGREEN,Energy,Renewable Energy,1.23,Large Cap
4,5,ADANIPORTS,Industrials,Infrastructure,2.12,Large Cap
5,6,ADANIPOWER,Energy,Power & Utilities,3.01,Large Cap
6,7,AMBUJACEM,Materials,Cement,2.82,Large Cap
7,8,APOLLOHOSP,Healthcare,Healthcare,0.62,Large Cap
8,9,ASIANPAINT,Materials,Paints & Coatings,1.13,Large Cap
9,10,ATGL,Energy,Gas Distribution,0.60,Large Cap


In [18]:
peer_groups["company_id"] = (
    peer_groups["company_id"]
    .astype(str)
    .str.strip()
    .str.upper()
)

peer_groups["peer_group_name"] = (
    peer_groups["peer_group_name"]
    .astype(str)
    .str.strip()
)

print("Peer Group Rows:", len(peer_groups))
print("Unique Companies:", peer_groups["company_id"].nunique())
print("Unique Peer Groups:", peer_groups["peer_group_name"].nunique())

display(peer_groups.head())

Peer Group Rows: 56
Unique Companies: 56
Unique Peer Groups: 11


,id,peer_group_name,company_id,is_benchmark
0,1,Private Banks,HDFCBANK,True
1,2,Private Banks,ICICIBANK,False
2,3,Private Banks,AXISBANK,False
3,4,Private Banks,KOTAKBANK,False
4,5,Private Banks,INDUSINDBK,False


In [19]:
sectors["company_id"] = (
    sectors["company_id"]
    .astype(str)
    .str.strip()
    .str.upper()
)

sectors["broad_sector"] = (
    sectors["broad_sector"]
    .astype(str)
    .str.strip()
)

sectors["sub_sector"] = (
    sectors["sub_sector"]
    .astype(str)
    .str.strip()
)

print("Sector Rows:", len(sectors))
print("Unique Companies:", sectors["company_id"].nunique())

display(sectors.head())

Sector Rows: 92
Unique Companies: 92


,id,company_id,broad_sector,sub_sector,index_weight_pct,market_cap_category
0,1,ABB,Industrials,Capital Goods,0.81,Large Cap
1,2,ADANIENSOL,Energy,Power & Utilities,0.65,Large Cap
2,3,ADANIENT,Industrials,Conglomerates,1.50,Large Cap
3,4,ADANIGREEN,Energy,Renewable Energy,1.23,Large Cap
4,5,ADANIPORTS,Industrials,Infrastructure,2.12,Large Cap


In [20]:
financial_ratios = pd.read_excel(
    "../data/raw/supporting/financial_ratios.xlsx"
)

print("FINANCIAL RATIOS")
print("Shape:", financial_ratios.shape)
print("Columns:")

for col in financial_ratios.columns:
    print(col)

display(financial_ratios.head())

FINANCIAL RATIOS
Shape: (1184, 16)
Columns:
id
company_id
year
net_profit_margin_pct
operating_profit_margin_pct
return_on_equity_pct
debt_to_equity
interest_coverage
asset_turnover
free_cash_flow_cr
capex_cr
earnings_per_share
book_value_per_share
dividend_payout_ratio_pct
total_debt_cr
cash_from_operations_cr


,id,company_id,year,net_profit_margin_pct,operating_profit_margin_pct,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,free_cash_flow_cr,capex_cr,earnings_per_share,book_value_per_share,dividend_payout_ratio_pct,total_debt_cr,cash_from_operations_cr
0,1,ABB,Dec 2012,8.77,12.0,22.41,0.0,NaN,1.8225,42.0,59.0,68.0,3.0810,25.0,0,101.0
1,2,ABB,Mar 2014,8.70,12.0,25.13,0.0,NaN,1.9982,11.0,144.0,93.0,3.7524,25.0,0,155.0
2,3,ABB,Mar 2014,8.70,12.0,25.13,0.0,NaN,1.9982,0.0,0.0,93.0,3.7524,25.0,0,0.0
3,4,ABB,Mar 2015,10.00,14.0,24.44,0.0,NaN,1.6659,28.0,187.0,108.0,4.4619,29.0,0,215.0
4,5,ABB,Mar 2015,10.00,14.0,24.44,0.0,NaN,1.6659,-1899.0,1864.0,108.0,4.4619,29.0,0,-35.0


In [21]:
print(financial_ratios.dtypes)

id                               int64
company_id                         str
year                               str
net_profit_margin_pct          float64
operating_profit_margin_pct    float64
return_on_equity_pct           float64
debt_to_equity                 float64
interest_coverage              float64
asset_turnover                 float64
free_cash_flow_cr              float64
capex_cr                       float64
earnings_per_share             float64
book_value_per_share           float64
dividend_payout_ratio_pct      float64
total_debt_cr                    int64
cash_from_operations_cr        float64
dtype: object


In [22]:
ratio_candidates = [
    col
    for col in financial_ratios.columns
    if any(
        word in str(col).lower()
        for word in [
            "roe",
            "roce",
            "margin",
            "debt",
            "equity",
            "coverage",
            "turnover",
            "ratio"
        ]
    )
]

print("Possible Ratio Columns:")

for col in ratio_candidates:
    print(col)

Possible Ratio Columns:
net_profit_margin_pct
operating_profit_margin_pct
return_on_equity_pct
debt_to_equity
interest_coverage
asset_turnover
dividend_payout_ratio_pct
total_debt_cr
cash_from_operations_cr


In [23]:
financial_ratios["company_id"] = (
    financial_ratios["company_id"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print(
    "Financial Ratio Companies:",
    financial_ratios["company_id"].nunique()
)

display(financial_ratios.head())

Financial Ratio Companies: 92


,id,company_id,year,net_profit_margin_pct,operating_profit_margin_pct,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,free_cash_flow_cr,capex_cr,earnings_per_share,book_value_per_share,dividend_payout_ratio_pct,total_debt_cr,cash_from_operations_cr
0,1,ABB,Dec 2012,8.77,12.0,22.41,0.0,NaN,1.8225,42.0,59.0,68.0,3.0810,25.0,0,101.0
1,2,ABB,Mar 2014,8.70,12.0,25.13,0.0,NaN,1.9982,11.0,144.0,93.0,3.7524,25.0,0,155.0
2,3,ABB,Mar 2014,8.70,12.0,25.13,0.0,NaN,1.9982,0.0,0.0,93.0,3.7524,25.0,0,0.0
3,4,ABB,Mar 2015,10.00,14.0,24.44,0.0,NaN,1.6659,28.0,187.0,108.0,4.4619,29.0,0,215.0
4,5,ABB,Mar 2015,10.00,14.0,24.44,0.0,NaN,1.6659,-1899.0,1864.0,108.0,4.4619,29.0,0,-35.0


In [24]:
peer_financials = peer_groups.merge(
    financial_ratios,
    on="company_id",
    how="inner"
)

print("PEER FINANCIAL DATA")
print("Shape:", peer_financials.shape)
print(
    "Unique Companies:",
    peer_financials["company_id"].nunique()
)

display(peer_financials.head(10))

PEER FINANCIAL DATA
Shape: (730, 19)
Unique Companies: 55


,id_x,peer_group_name,company_id,is_benchmark,id_y,year,net_profit_margin_pct,operating_profit_margin_pct,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,free_cash_flow_cr,capex_cr,earnings_per_share,book_value_per_share,dividend_payout_ratio_pct,total_debt_cr,cash_from_operations_cr
0,1,Private Banks,HDFCBANK,True,479,Mar 2013,19.25,3534.0,18.84,9.1583,1.7722,0.0880,-6749.0,902.0,14.0,7.6981,19.0,335588,-5847.0
1,1,Private Banks,HDFCBANK,True,480,Mar 2014,20.60,5602.0,19.85,9.4341,1.6294,0.0845,3112.0,1099.0,18.0,9.2015,19.0,416677,4211.0
2,1,Private Banks,HDFCBANK,True,481,Mar 2015,21.12,7214.0,16.95,8.0717,1.6947,0.0835,-22081.0,800.0,21.0,12.6056,19.0,509762,-21281.0
3,1,Private Banks,HDFCBANK,True,482,Mar 2016,20.30,9037.0,17.25,8.7423,1.7900,0.0829,-35272.0,837.0,25.0,14.6846,19.0,649587,-34435.0
4,1,Private Banks,HDFCBANK,True,483,Mar 2017,20.90,11374.0,16.69,8.0784,1.8498,0.0821,16136.0,1146.0,30.0,17.8936,18.0,741550,17282.0
5,1,Private Banks,HDFCBANK,True,484,Mar 2018,21.76,13374.0,16.94,8.6207,1.8402,0.0773,16372.0,842.0,36.0,21.1173,18.0,944817,17214.0
6,1,Private Banks,HDFCBANK,True,485,Mar 2019,21.34,16592.0,14.61,7.0294,1.8405,0.0813,-64375.0,1503.0,41.0,28.1969,18.0,1080235,-62872.0
7,1,Private Banks,HDFCBANK,True,486,Mar 2020,22.34,14593.0,15.48,7.5587,1.8277,0.0773,-18272.0,1403.0,50.0,32.1821,5.0,1333041,-16869.0
8,1,Private Banks,HDFCBANK,True,487,Mar 2021,24.78,16848.0,15.18,7.2037,1.9197,0.0714,40653.0,1823.0,58.0,38.0780,11.0,1511418,42476.0
9,1,Private Banks,HDFCBANK,True,488,Mar 2022,28.07,20795.0,15.43,7.2170,1.7813,0.0640,-14011.0,2051.0,69.0,44.5634,23.0,1784970,-11960.0


In [25]:
peer_financials = peer_financials.merge(
    sectors[
        [
            "company_id",
            "broad_sector",
            "sub_sector",
            "market_cap_category"
        ]
    ],
    on="company_id",
    how="left"
)

print("FINAL PEER DATA SHAPE:")
print(peer_financials.shape)

display(
    peer_financials[
        [
            "company_id",
            "peer_group_name",
            "broad_sector",
            "sub_sector",
            "market_cap_category"
        ]
    ].head(10)
)

FINAL PEER DATA SHAPE:
(730, 22)


,company_id,peer_group_name,broad_sector,sub_sector,market_cap_category
0,HDFCBANK,Private Banks,Financials,Private Banks,Large Cap
1,HDFCBANK,Private Banks,Financials,Private Banks,Large Cap
2,HDFCBANK,Private Banks,Financials,Private Banks,Large Cap
3,HDFCBANK,Private Banks,Financials,Private Banks,Large Cap
4,HDFCBANK,Private Banks,Financials,Private Banks,Large Cap
5,HDFCBANK,Private Banks,Financials,Private Banks,Large Cap
6,HDFCBANK,Private Banks,Financials,Private Banks,Large Cap
7,HDFCBANK,Private Banks,Financials,Private Banks,Large Cap
8,HDFCBANK,Private Banks,Financials,Private Banks,Large Cap
9,HDFCBANK,Private Banks,Financials,Private Banks,Large Cap


In [26]:
exclude_cols = [
    "id",
    "company_id",
    "peer_group_name",
    "is_benchmark",
    "broad_sector",
    "sub_sector",
    "market_cap_category"
]

numeric_cols = (
    peer_financials
    .select_dtypes(include="number")
    .columns
    .tolist()
)

numeric_cols = [
    col
    for col in numeric_cols
    if col not in exclude_cols
]

print("Numeric Columns Used for Ranking:")

for col in numeric_cols:
    print(col)

Numeric Columns Used for Ranking:
id_x
id_y
net_profit_margin_pct
operating_profit_margin_pct
return_on_equity_pct
debt_to_equity
interest_coverage
asset_turnover
free_cash_flow_cr
capex_cr
earnings_per_share
book_value_per_share
dividend_payout_ratio_pct
total_debt_cr
cash_from_operations_cr


In [27]:
id_columns = [
    col
    for col in peer_financials.columns
    if col == "id"
    or col.startswith("id_")
]

print("ID Columns:", id_columns)

ID Columns: ['id_x', 'id_y']


In [28]:
percentile_columns = []

for col in numeric_cols:

    percentile_col = f"{col}_peer_percentile"

    peer_financials[percentile_col] = (
        peer_financials
        .groupby("peer_group_name")[col]
        .rank(
            pct=True,
            method="average"
        )
        * 100
    )

    percentile_columns.append(percentile_col)

print("Created Percentile Columns:")

for col in percentile_columns:
    print(col)

Created Percentile Columns:
id_x_peer_percentile
id_y_peer_percentile
net_profit_margin_pct_peer_percentile
operating_profit_margin_pct_peer_percentile
return_on_equity_pct_peer_percentile
debt_to_equity_peer_percentile
interest_coverage_peer_percentile
asset_turnover_peer_percentile
free_cash_flow_cr_peer_percentile
capex_cr_peer_percentile
earnings_per_share_peer_percentile
book_value_per_share_peer_percentile
dividend_payout_ratio_pct_peer_percentile
total_debt_cr_peer_percentile
cash_from_operations_cr_peer_percentile


In [29]:
preview_cols = [
    "company_id",
    "peer_group_name"
]

preview_cols += numeric_cols[:3]
preview_cols += percentile_columns[:3]

display(
    peer_financials[
        preview_cols
    ].head(20)
)

,company_id,peer_group_name,id_x,id_y,net_profit_margin_pct,id_x_peer_percentile,id_y_peer_percentile,net_profit_margin_pct_peer_percentile
0,HDFCBANK,Private Banks,1,479,19.25,10.833333,21.666667,35.000000
1,HDFCBANK,Private Banks,1,480,20.60,10.833333,23.333333,50.000000
2,HDFCBANK,Private Banks,1,481,21.12,10.833333,25.000000,60.000000
3,HDFCBANK,Private Banks,1,482,20.30,10.833333,26.666667,46.666667
4,HDFCBANK,Private Banks,1,483,20.90,10.833333,28.333333,56.666667
5,HDFCBANK,Private Banks,1,484,21.76,10.833333,30.000000,63.333333
6,HDFCBANK,Private Banks,1,485,21.34,10.833333,31.666667,61.666667
7,HDFCBANK,Private Banks,1,486,22.34,10.833333,33.333333,65.000000
8,HDFCBANK,Private Banks,1,487,24.78,10.833333,35.000000,81.666667
9,HDFCBANK,Private Banks,1,488,28.07,10.833333,36.666667,90.000000


In [30]:
benchmark_companies = peer_financials[
    peer_financials["is_benchmark"] == True
]

print("BENCHMARK COMPANIES")
print(
    "Count:",
    benchmark_companies["company_id"].nunique()
)

display(
    benchmark_companies[
        [
            "company_id",
            "peer_group_name",
            "is_benchmark"
        ]
    ].drop_duplicates()
)

BENCHMARK COMPANIES
Count: 10


,company_id,peer_group_name,is_benchmark
0,HDFCBANK,Private Banks,True
144,TCS,IT Services,True
215,SUNPHARMA,Pharmaceuticals,True
275,MARUTI,Automobiles,True
370,LICI,Life Insurance,True
412,RELIANCE,Oil & Gas,True
472,NTPC,Power & Utilities,True
564,TATASTEEL,Steel,True
612,HINDUNILVR,FMCG,True
696,BAJFINANCE,Consumer Finance,True


In [31]:
rank_columns = []

for col in numeric_cols:

    rank_col = f"{col}_peer_rank"

    peer_financials[rank_col] = (
        peer_financials
        .groupby("peer_group_name")[col]
        .rank(
            ascending=False,
            method="dense"
        )
    )

    rank_columns.append(rank_col)

print("Peer Rank Columns Created:")

for col in rank_columns:
    print(col)

Peer Rank Columns Created:
id_x_peer_rank
id_y_peer_rank
net_profit_margin_pct_peer_rank
operating_profit_margin_pct_peer_rank
return_on_equity_pct_peer_rank
debt_to_equity_peer_rank
interest_coverage_peer_rank
asset_turnover_peer_rank
free_cash_flow_cr_peer_rank
capex_cr_peer_rank
earnings_per_share_peer_rank
book_value_per_share_peer_rank
dividend_payout_ratio_pct_peer_rank
total_debt_cr_peer_rank
cash_from_operations_cr_peer_rank


In [32]:
peer_financials["peer_group_size"] = (
    peer_financials
    .groupby("peer_group_name")["company_id"]
    .transform("nunique")
)

display(
    peer_financials[
        [
            "company_id",
            "peer_group_name",
            "peer_group_size"
        ]
    ].drop_duplicates().head(20)
)

,company_id,peer_group_name,peer_group_size
0,HDFCBANK,Private Banks,5
12,ICICIBANK,Private Banks,5
24,AXISBANK,Private Banks,5
36,KOTAKBANK,Private Banks,5
48,INDUSINDBK,Private Banks,5
60,BANKBARODA,Public Sector Banks,3
72,CANBK,Public Sector Banks,3
84,PNB,Public Sector Banks,3
144,TCS,IT Services,5
156,INFY,IT Services,5


In [33]:
peer_financials["overall_peer_percentile"] = (
    peer_financials[
        percentile_columns
    ]
    .mean(
        axis=1,
        skipna=True
    )
)

display(
    peer_financials[
        [
            "company_id",
            "peer_group_name",
            "overall_peer_percentile"
        ]
    ]
    .sort_values(
        "overall_peer_percentile",
        ascending=False
    )
    .head(20)
)

,company_id,peer_group_name,overall_peer_percentile
155,TCS,IT Services,78.624253
151,TCS,IT Services,76.148673
585,JSWSTEEL,Steel,75.277778
505,POWERGRID,Power & Utilities,74.855072
152,TCS,IT Services,74.788953
504,POWERGRID,Power & Utilities,74.782609
501,POWERGRID,Power & Utilities,74.492754
500,POWERGRID,Power & Utilities,74.420290
507,POWERGRID,Power & Utilities,74.130435
506,POWERGRID,Power & Utilities,74.057971


In [34]:
peer_financials["overall_peer_rank"] = (
    peer_financials
    .groupby("peer_group_name")[
        "overall_peer_percentile"
    ]
    .rank(
        ascending=False,
        method="dense"
    )
)

display(
    peer_financials[
        [
            "company_id",
            "peer_group_name",
            "overall_peer_percentile",
            "overall_peer_rank",
            "peer_group_size"
        ]
    ]
    .sort_values(
        [
            "peer_group_name",
            "overall_peer_rank"
        ]
    )
    .head(30)
)

,company_id,peer_group_name,overall_peer_percentile,overall_peer_rank,peer_group_size
357,HEROMOTOCO,Automobiles,70.526316,1.0,7
353,HEROMOTOCO,Automobiles,70.000000,2.0,7
351,HEROMOTOCO,Automobiles,68.245614,3.0,7
350,HEROMOTOCO,Automobiles,68.035088,4.0,7
349,HEROMOTOCO,Automobiles,67.228070,5.0,7
354,HEROMOTOCO,Automobiles,66.000000,6.0,7
352,HEROMOTOCO,Automobiles,64.491228,7.0,7
347,HEROMOTOCO,Automobiles,64.140351,8.0,7
298,TATAMOTORS,Automobiles,63.403509,9.0,7
334,BAJAJ-AUTO,Automobiles,62.877193,10.0,7


In [35]:
def classify_peer_performance(percentile):

    if pd.isna(percentile):
        return "Insufficient Data"

    elif percentile >= 80:
        return "Peer Leader"

    elif percentile >= 60:
        return "Above Peer Average"

    elif percentile >= 40:
        return "Peer Average"

    elif percentile >= 20:
        return "Below Peer Average"

    else:
        return "Peer Laggard"


peer_financials["peer_performance_class"] = (
    peer_financials[
        "overall_peer_percentile"
    ]
    .apply(classify_peer_performance)
)

display(
    peer_financials[
        [
            "company_id",
            "peer_group_name",
            "overall_peer_percentile",
            "overall_peer_rank",
            "peer_performance_class"
        ]
    ]
    .sort_values(
        "overall_peer_percentile",
        ascending=False
    )
    .head(20)
)

,company_id,peer_group_name,overall_peer_percentile,overall_peer_rank,peer_performance_class
155,TCS,IT Services,78.624253,1.0,Above Peer Average
151,TCS,IT Services,76.148673,2.0,Above Peer Average
585,JSWSTEEL,Steel,75.277778,1.0,Above Peer Average
505,POWERGRID,Power & Utilities,74.855072,1.0,Above Peer Average
152,TCS,IT Services,74.788953,3.0,Above Peer Average
504,POWERGRID,Power & Utilities,74.782609,2.0,Above Peer Average
501,POWERGRID,Power & Utilities,74.492754,3.0,Above Peer Average
500,POWERGRID,Power & Utilities,74.420290,4.0,Above Peer Average
507,POWERGRID,Power & Utilities,74.130435,5.0,Above Peer Average
506,POWERGRID,Power & Utilities,74.057971,6.0,Above Peer Average


In [36]:
peer_report_cols = [
    "company_id",
    "peer_group_name",
    "is_benchmark",
    "broad_sector",
    "sub_sector",
    "market_cap_category",
    "peer_group_size",
    "overall_peer_percentile",
    "overall_peer_rank",
    "peer_performance_class"
]

peer_comparison_report = (
    peer_financials[
        peer_report_cols
    ]
    .drop_duplicates()
    .sort_values(
        [
            "peer_group_name",
            "overall_peer_rank"
        ]
    )
)

peer_comparison_report.to_csv(
    "../reports/day18_peer_comparison.csv",
    index=False
)

print("Saved: day18_peer_comparison.csv")

display(peer_comparison_report.head(20))

Saved: day18_peer_comparison.csv


,company_id,peer_group_name,is_benchmark,broad_sector,sub_sector,market_cap_category,peer_group_size,overall_peer_percentile,overall_peer_rank,peer_performance_class
357,HEROMOTOCO,Automobiles,False,Consumer Discretionary,Two Wheelers,Large Cap,7,70.526316,1.0,Above Peer Average
353,HEROMOTOCO,Automobiles,False,Consumer Discretionary,Two Wheelers,Large Cap,7,70.000000,2.0,Above Peer Average
351,HEROMOTOCO,Automobiles,False,Consumer Discretionary,Two Wheelers,Large Cap,7,68.245614,3.0,Above Peer Average
350,HEROMOTOCO,Automobiles,False,Consumer Discretionary,Two Wheelers,Large Cap,7,68.035088,4.0,Above Peer Average
349,HEROMOTOCO,Automobiles,False,Consumer Discretionary,Two Wheelers,Large Cap,7,67.228070,5.0,Above Peer Average
354,HEROMOTOCO,Automobiles,False,Consumer Discretionary,Two Wheelers,Large Cap,7,66.000000,6.0,Above Peer Average
352,HEROMOTOCO,Automobiles,False,Consumer Discretionary,Two Wheelers,Large Cap,7,64.491228,7.0,Above Peer Average
347,HEROMOTOCO,Automobiles,False,Consumer Discretionary,Two Wheelers,Large Cap,7,64.140351,8.0,Above Peer Average
298,TATAMOTORS,Automobiles,False,Consumer Discretionary,Automobiles,Large Cap,7,63.403509,9.0,Above Peer Average
334,BAJAJ-AUTO,Automobiles,False,Consumer Discretionary,Two Wheelers,Large Cap,7,62.877193,10.0,Above Peer Average


In [37]:
detailed_cols = (
    [
        "company_id",
        "peer_group_name"
    ]
    + numeric_cols
    + percentile_columns
    + rank_columns
)

detailed_peer_report = (
    peer_financials[
        detailed_cols
    ]
)

detailed_peer_report.to_csv(
    "../reports/day18_peer_percentiles.csv",
    index=False
)

print("Saved: day18_peer_percentiles.csv")

print(
    "Report Shape:",
    detailed_peer_report.shape
)

Saved: day18_peer_percentiles.csv
Report Shape: (730, 47)


In [38]:
peer_leaders = peer_comparison_report[
    peer_comparison_report[
        "peer_performance_class"
    ] == "Peer Leader"
]

peer_leaders.to_csv(
    "../reports/day18_peer_leaders.csv",
    index=False
)

print("PEER LEADERS")
print("Count:", len(peer_leaders))

display(peer_leaders)

PEER LEADERS
Count: 0


,company_id,peer_group_name,is_benchmark,broad_sector,sub_sector,market_cap_category,peer_group_size,overall_peer_percentile,overall_peer_rank,peer_performance_class


In [39]:
benchmark_report = peer_comparison_report[
    peer_comparison_report["is_benchmark"] == True
]

benchmark_report.to_csv(
    "../reports/day18_benchmark_peer_performance.csv",
    index=False
)

print("BENCHMARK PERFORMANCE")

display(benchmark_report)

BENCHMARK PERFORMANCE


,company_id,peer_group_name,is_benchmark,broad_sector,sub_sector,market_cap_category,peer_group_size,overall_peer_percentile,overall_peer_rank,peer_performance_class
279,MARUTI,Automobiles,True,Consumer Discretionary,Automobiles,Mid Cap,7,59.157895,14.0,Peer Average
285,MARUTI,Automobiles,True,Consumer Discretionary,Automobiles,Mid Cap,7,56.526316,23.0,Peer Average
286,MARUTI,Automobiles,True,Consumer Discretionary,Automobiles,Mid Cap,7,56.421053,24.0,Peer Average
280,MARUTI,Automobiles,True,Consumer Discretionary,Automobiles,Mid Cap,7,55.964912,25.0,Peer Average
281,MARUTI,Automobiles,True,Consumer Discretionary,Automobiles,Mid Cap,7,55.578947,27.0,Peer Average
...,...,...,...,...,...,...,...,...,...,...
575,TATASTEEL,Steel,True,Materials,Steel,Mid Cap,4,43.888889,34.0,Peer Average
566,TATASTEEL,Steel,True,Materials,Steel,Mid Cap,4,40.069444,36.0,Peer Average
564,TATASTEEL,Steel,True,Materials,Steel,Mid Cap,4,39.097222,37.0,Below Peer Average
567,TATASTEEL,Steel,True,Materials,Steel,Mid Cap,4,37.569444,39.0,Below Peer Average


In [40]:
print("=" * 50)
print("DAY 18 FINAL VALIDATION")
print("=" * 50)

print(
    "Companies Analysed:",
    peer_comparison_report["company_id"].nunique()
)

print(
    "Peer Groups:",
    peer_comparison_report["peer_group_name"].nunique()
)

print(
    "Percentile Metrics:",
    len(percentile_columns)
)

print(
    "Peer Leaders:",
    len(peer_leaders)
)

print(
    "Benchmark Companies:",
    benchmark_report["company_id"].nunique()
)

print("\nPerformance Distribution:")

print(
    peer_comparison_report[
        "peer_performance_class"
    ].value_counts()
)

print("\nDAY 18 COMPLETED SUCCESSFULLY")

DAY 18 FINAL VALIDATION
Companies Analysed: 55
Peer Groups: 11
Percentile Metrics: 15
Peer Leaders: 0
Benchmark Companies: 10

Performance Distribution:
peer_performance_class
Peer Average          500
Above Peer Average    122
Below Peer Average    104
Name: count, dtype: int64

DAY 18 COMPLETED SUCCESSFULLY


## Day 18 Summary

### Peer Comparison & Percentile Ranking

- Loaded official peer group and sector supporting datasets.
- Mapped companies to predefined peer groups.
- Integrated peer companies with financial ratio data.
- Calculated peer-level percentile rankings for numeric financial metrics.
- Generated metric-level peer ranks.
- Calculated overall peer percentile and peer rank.
- Classified companies as Peer Leader, Above Peer Average, Peer Average, Below Peer Average, or Peer Laggard.
- Evaluated benchmark companies against their respective peer groups.
- Exported peer comparison, detailed percentile, peer leader, and benchmark performance reports.

### Output Reports

- `day18_peer_comparison.csv`
- `day18_peer_percentiles.csv`
- `day18_peer_leaders.csv`
- `day18_benchmark_peer_performance.csv`

**Day 18 Status: Completed**